# Data Description
- Background: 第六輪來源保存：188個履歷出處HTTP嘗試、61個首長名錄／政府頁嘗試與11個候選人介紹頁。HTTP 200不等於個人背景已核實。原始HTTP bytes保留；manifest及retrieval_plan為本地擷取／查找中介資料，不冒充原網站JSON。
- date received: 20261019（本次實際擷取／保存日期）
- Path to data file: 下列精確manifest記錄來源URL及儲存檔案；每份HTTP原文的來源URL保留於manifest。
- Unit of observation: manifest一列為一次來源URL擷取紀錄；body檔為該次HTTP bytes。retrieval_plan一列為第三方履歷出處線索。
- Sample period: 來源包含歷史選舉、公報與人物報導，以及2026當屆競選頁；不是統一觀察窗口。
- Known issues: 錯誤、403、406、空檔與連線失敗保留；未繞過登入或存取限制。JSON中的原始主張、姓名命中、官網HTTP成功均不構成學位或身分驗證。雲林政府頁未取得可用直接官方內文，首長比對只列名錄來源。宜蘭民選首長與代理首長分開記錄。其他縣市未見代理名稱不等同已證明沒有代理。
- Definition for each variable: `url`原始請求URL；`status`/`http_status`實際HTTP狀態或連線失敗碼；`content_type`回應MIME；`path`儲存位置；`sha256`儲存bytes雜湊；`final_url`重新導向終點；`error`擷取錯誤；`captured_date`實際記錄日。少數舊紀錄未列完整metadata，保留缺項。查找計畫的upstream欄位為第三方主張，不是本研究核實值。
- Data path: 所有實際具名檔案與預先核對SHA-256列於下方PINS；manifest為以下3份。
- Role: raw_http_capture／retrieval_metadata
- Verification status: 執行下方檔案存在、邊界與雜湊檢查；結果只驗證保存檔身分，不認證內容正確。
- SHA-256: 下方PINS為執行前固定值。
- Next processing notebook: [第六輪處理](../code/career_round6_20261019.ipynb)

### Manifest files
- `data/raw/social_followup_round6_20261019/manifest.json` — `f5471133039402a9e21143f23d30b70b645ddf3406805e19f78764f5b2063746`
- `data/raw/career_expansion_20261019/manifest.json` — `5aa5cc26f08c1d327b2273e10aa43ad4a17dc610ad201de17008df23d8418679`
- `data/raw/mayor_incumbency_20261019/manifest.json` — `cb759787a01c437043c1ddca11397b6392643ebced7c74784856ba4e0e0161af`

新增接用已於2026-10-17保存的中選會原始歷史表：2014直轄市區長、2018直轄市區民代表、2020區域立委、2024平地立委，來源檔與當時解壓manifest的hash均固定於PINS。本輪另行讀取，不宣稱重新下載；官方性別代碼1=男、2=女，民國生日轉西元後檢查實際日期，並以姓名、縣市、參選公職／年份與當屆人物頁交叉支持，排除其他林啟明同名者。

### 續補使用的既有來源與擷取失敗

另讀已保存的陳重嘉官方HTML與温世政2021專業學會PDF，後者已視讀第1頁第8列的生日欄。另兩筆2026媒體頁直接擷取回403，原body保存於mayor_birth_followup_20261019；web_tool_excerpts.json是實際透過web工具open/find讀取後的短摘錄，非原始HTML，與失敗HTTP分開記錄。原260次擷取加這2筆失敗共262筆；所有新增原檔及方法metadata固定hash後讀取。未繞過403或登入限制。

公開來源副本修訂（2026-10-19）：7份第三方HTML中的Google地圖API key已遮蔽；原始HTTP回應bytes已核對後私存於Git儲存庫外。來源manifest同時保存original_response_sha256與清理後sha256，PINS及輸出來源指紋綁定清理後公開版本。人物學經歷、生日、政見等研究文字未刪改。詳見[清理紀錄](../../report/source_capture_security_20261019.md)。

In [1]:
from pathlib import Path
import pandas as pd,json,hashlib
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"AGENTS.md").is_file() and (p/"templates").is_dir())
PINS={'data/raw/social_followup_round6_20261019/manifest.json': 'f5471133039402a9e21143f23d30b70b645ddf3406805e19f78764f5b2063746', 'data/raw/career_expansion_20261019/manifest.json': '5aa5cc26f08c1d327b2273e10aa43ad4a17dc610ad201de17008df23d8418679', 'data/raw/mayor_incumbency_20261019/manifest.json': 'cb759787a01c437043c1ddca11397b6392643ebced7c74784856ba4e0e0161af', 'data/raw/career_expansion_20261019/retrieval_plan.json': 'fa053b850dc006d926c9141931a2d06f68a1b304cca92a740e8c30c405537e1c', 'data/raw/mayor_incumbency_20261019/profile_plan.json': 'f7d464b7e869b2753217a33804b0e3be74377aabff7c5c2ac090ad1b8a9b470a', 'data/raw/mayor_incumbency_20261019/government_plan.json': 'd59e09dc1c3d3d04fb7b2d0ca74f3853b748125995f0a742529ea3d6868ae938', 'data/raw/social_followup_round6_20261019/c9fb4f96a8a3579957d9.html': 'fefedba85d34ebf1141090f3447b9f3f83cde424d767cb60b2e70d4fc5a5e350', 'data/raw/social_followup_round6_20261019/fff69c0c06e8ba50d093.html': '9079d0f4f0094d3e4bda4d2e2a6aae8f4346b6484131b0ee2f962c8b2db1ae19', 'data/raw/social_followup_round6_20261019/b39ba23e64e7bb9ca8b3.html': 'c0a7c18159f0dc6e66dfcca6757183d0eb26f66487871cd5fb635c1c4e72bdd6', 'data/raw/social_followup_round6_20261019/ce69d6db1d9d4ba73f69.html': '1a3df0dfd62ff0b4daad24f25f439bbeb356876d7149fdba87f9f1c06bb1a964', 'data/raw/social_followup_round6_20261019/0277960ed4f266884bad.html': 'f31d0165ee321abb26ce8b9cb6816b9dbbd4655fc9a7f9ddc309a75bcbc12c90', 'data/raw/social_followup_round6_20261019/048db667949e381a9804.html': 'f056fade24662549e0c320922485f61994193d4487549cf5776eb49fa251a6ec', 'data/raw/social_followup_round6_20261019/a201e95cde4d5fb57eea.html': 'b026c9a3a665849da4bbbf91567db46f9b963d643063625084ebfd46ba028982', 'data/raw/social_followup_round6_20261019/f6a9400f7efa13e8328f.html': 'fd1cfffe4b4cd07620beb79f0df3f50904e6bdb5bff6cfa9fc3e011f857d1b33', 'data/raw/social_followup_round6_20261019/f503253c8ee50fc342aa.html': 'ac5547431eee8a69c13a8bd3f637710269067de70c7ab92c96399f21333e27c7', 'data/raw/social_followup_round6_20261019/dd2a14a1b62b3140824a.html': 'b5ef4b7d0a3e0ae42193ec8edb83cd795d8d30330b1236f51066ea307ff65587', 'data/raw/social_followup_round6_20261019/ce6069c72177c68a288f.html': 'fe2d01540f6bf21f5873b8fffc641acb1891b26b3a697ee3bf65a5e63eb79351', 'data/raw/career_expansion_20261019/d7a9d13bd3956f33b02d.source': '822c5eddd92f06bfcc349b20db46ec32cf0381e474f6fcf355ab57a336fc9ca2', 'data/raw/career_expansion_20261019/fa3e048ef6ab320c6b5a.source': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'data/raw/career_expansion_20261019/0b3415737e463fb430c1.source': 'c84595c85e495bc5446e06d8e30c7dc398ed51d9118ff3e745b9af4faedeaf23', 'data/raw/career_expansion_20261019/48e23f089c402141d148.source': 'bf287c7834b4945d08907f1b10bfbf8303490eac784bc9df49bfa94e0b7700ce', 'data/raw/career_expansion_20261019/135bbd297188cb4aded2.source': 'ce9550cd344301cdd7b26b2dc920a704b058f4a29fe12e412667e1847add6072', 'data/raw/career_expansion_20261019/f16c5a101610f6df2a8f.source': '12b62665a98fcb881e123b50c109bb5a9ba124147fb790613d30436690fbf44d', 'data/raw/career_expansion_20261019/fae2f34650790f53757f.source': '1fcc90869dea5464b97bf5539697d170431721e7b7025d954beb002f94f46517', 'data/raw/career_expansion_20261019/1cbf0002574d9944ccfa.source': '15cc48bfb18efcf47f5bdd50b628277a4dca522666f9b894dc09fce32d0ca78f', 'data/raw/career_expansion_20261019/82a67387cbeb1a068bc3.source': 'd05f43b367313c85806768540aba9c9bd26dff95abb9cae5ec8fd4db91da1678', 'data/raw/career_expansion_20261019/2ae9c291e658d21ff79d.source': 'e3dce092381f281da6be2e15fd725bf5f1e03fe151ce389184f4129767a1d8b2', 'data/raw/career_expansion_20261019/2a06c97bbf3ad203c121.source': '1410168b31a6f32c6fa9225da2665fefca552fae0bca3139f7245331e398d1de', 'data/raw/career_expansion_20261019/766132a7f7455c52499f.source': 'eab09f82e87641fb3e8c72b6c9e061f11684c7d8345925aad1fd7de4c05f87ab', 'data/raw/career_expansion_20261019/9a667727ce0ff9a5c11f.source': '41d33d681c2a2e5449d3632b544ce5d422bcb5e5f3814ba48be6cbeffbdac519', 'data/raw/career_expansion_20261019/b33ed4a59c34d3f9b988.source': '8250ad128e8b103d4d67787919d405ecb6eefc1f90af52763fe8b30fb6d29571', 'data/raw/career_expansion_20261019/468a3c6dc5fb16dabf7f.source': '04909d5a22ef0b0728a0813e8fd0cb44385baf4fb605a1959268ff14e78758e7', 'data/raw/career_expansion_20261019/2b21ad812d5c445a1a7b.source': '707168cd900f36721b2bf8ea76fd9c2928d1f1941647ccff0d7ae5f8a0d37a78', 'data/raw/career_expansion_20261019/52d517f2cd091856f373.source': '3385e40ba0bbe55a1ff6438be2368ec25e1b1f39d49c4b4518dc3916d13a7230', 'data/raw/career_expansion_20261019/636eaf3f4ed4d5ea56c7.source': 'd2687658e07bc5a4df8f78f53477f621edf1c22a3fab797a9a3efa15d38c1355', 'data/raw/career_expansion_20261019/461875bbdfedf21ab43a.source': 'e59c0ab98f6e6cb36b877bb93752a2b495be0fdc997bb2de0f77246ce2feff46', 'data/raw/career_expansion_20261019/6ae3043edb1543535cf3.source': '3f17d9e07af08f3cddb8261b5ccba8a9d9b4042e1ccab89e3e73552b51759328', 'data/raw/career_expansion_20261019/3b06238b217252a8328e.source': '2bf0e9bd712821ceabd6b0d5cb5e09704151fe339f5130ad1089c6af88563850', 'data/raw/career_expansion_20261019/b07d1c2017f3a9ac0eb1.source': '11a2238baa8c48a21797b3f71ce418eb0cd2053a59a8ef81386bf5e70501fdc9', 'data/raw/career_expansion_20261019/feefa41d2dbcd3b7b3fb.source': 'a42de88e053b934e8a23288584ab4b1852876ce4a95aeca12b4b4f2fa2ab0445', 'data/raw/career_expansion_20261019/550cb9619ef257199cb1.source': 'b4ff51c960a83794504824f122fc43b93cb7dc05a9f227d4e4cccb0e496b6b9f', 'data/raw/career_expansion_20261019/beeec695342a2cbc917b.source': 'e090e5c9a14079ec46f3def19cfdd1c2dd6869c23724589ef97b7fc0c3d1fb90', 'data/raw/career_expansion_20261019/f0ccfe475c8a37b154f6.source': 'cdbb8fe941687db5571ba91d97cdeb55bcdf883a150bc53d1a147f5be91f57d6', 'data/raw/career_expansion_20261019/92d7983fa58a330e792d.source': 'c449936cf05a6ee8c47f66e129e0a9982db0028159c6296da64fcf11f0aca45f', 'data/raw/career_expansion_20261019/27fcdc4853f3a7022e5e.source': 'd7183092f2ae0d203c9f215fcaa75ba841734124852fa4c65599a57a8df15c1e', 'data/raw/career_expansion_20261019/5915299bddf0cff42d55.source': '7c2b4a59cce27a8382114afc1070ff8b025c08ba4e4f5cd1a05f53130c29c062', 'data/raw/career_expansion_20261019/333b4ea84bdeeb6bc421.source': 'a6ee552edc3d81a9dd335a014e8b26d0b42286966bc58fe9ee560df779087f76', 'data/raw/career_expansion_20261019/14f7ae4b1c9ec9dfd31d.source': '4a1cfb93ee7ba675932b54997f739d46a1ec2cc8e8bcf3ac8eece42e917299d5', 'data/raw/career_expansion_20261019/1292f72f4b4bdcc6aea6.source': 'ed4c9aff32fed8d8c99436ee750b25ec5cbcbb8ad81e207c52cece3360264be2', 'data/raw/career_expansion_20261019/da9be129aca5585ca016.source': '0a34266082c732d8d8d0deeb490784857177503f9c9e174062fc691db5ef4516', 'data/raw/career_expansion_20261019/de674628c3958af6cded.source': 'e5d6d1eff794a74bb25cc9cc107317c879d8dcc625d993ffc63e838a78e22eeb', 'data/raw/career_expansion_20261019/141d6752eda0a7501de2.source': 'b667e21669a0e09a7010e43a8cc0f471d0bf7c1ed488cc9691f9b535da6d722c', 'data/raw/career_expansion_20261019/36b6cff4de94c383f606.source': 'abc2d2026adb6f4d33a6542adbf32517e8e8b1ad6b5273e799c8493620b53881', 'data/raw/career_expansion_20261019/1f36d344b11d57d646ba.source': 'e7393761e6ad5a83c0287f9b11dcd218bbb016000ac6a0b80226e1db5106f697', 'data/raw/career_expansion_20261019/e5eab2c1ac86cf158208.source': '47af9a551c664055fd5c2dcf913e2c8cbaa07ffaf4012063966de7964054896a', 'data/raw/career_expansion_20261019/741b03060a0f6e7317d6.source': '1adfe4ee58956a85d430e9df5e85308905b76dbffd62e32f3b96b4b059887ef1', 'data/raw/career_expansion_20261019/df87732d3b2de62ca558.source': '61333bcb207e896e78d7a4c1417aa3b2bd6c7d1f727bea8b477ca2524b172dc5', 'data/raw/career_expansion_20261019/83791d2524e507c9be8a.source': 'd2a9b76c22709ca527ff4963e14aa058742bc9135ef017121449ec0eae0a7e9c', 'data/raw/career_expansion_20261019/26471c6d25da30794f5f.source': '9e0e6a9952e9469fda903160900fd01ff7dae6b89b8f672d29a947ac9cd84ee7', 'data/raw/career_expansion_20261019/b4eadb3e4d43bbd6fd18.source': '84734796727bafc56a14ede44ca924236206e2b04d134975b1f86b5901c8c3e5', 'data/raw/career_expansion_20261019/05fcf449ed72a84d3d22.source': '26a96092232dc3d8d9aee5938c94bd8f49a40d836a278b09d79c91e0454f01de', 'data/raw/career_expansion_20261019/4099da618c8b956be915.source': '07ff054abfb2d61bcbae24cc5307faf385d3f2d659a8eb188a09b5f104aa9c5a', 'data/raw/career_expansion_20261019/23cd9c77f9da99ea58ea.source': 'bfcb89b8fddea9efc03ed638aad63b649b24883bcd7efef786715201374aa929', 'data/raw/career_expansion_20261019/812263ce7bcc612dfcda.source': 'c2f5f2c8e6b902cf72cd7caf2cb5338816f241d4ba7ef6d14a558c4d7b2bb9f6', 'data/raw/career_expansion_20261019/1cc60e03026b92a723bc.source': '506f6c2c44bb8f02d3ee6cbbdd654ff5f5e26797e28a8e8f64618d1e9eabcbf0', 'data/raw/career_expansion_20261019/70699a479d5e3860cfa7.source': 'd318531466e6ab7dd9ee2752b15619a2bf507d9210cb22ef93b26b3333b819ec', 'data/raw/career_expansion_20261019/89aa472fd85ce71f998c.source': 'de271d878d11401fd9189e3def289780b0c77454f006e32f1ed5db2cbfff583f', 'data/raw/career_expansion_20261019/a19b41ac80955e0de263.source': '2a7e3299e59cc2fe1534fe7a89dffaf3711a8dc8724d72fae066cdec3119fd8c', 'data/raw/career_expansion_20261019/4c2ae41d29aaf3fcc6bf.source': 'c30f7885bc6c46fda2a962066ed9223774cd9802fe4732ee26eb00aeadf10f9d', 'data/raw/career_expansion_20261019/12dbc552dff5ff2f4929.source': 'e4384b274ee71eacd809081befc21074e3806b0759f2bf9891e3dbfe063becef', 'data/raw/career_expansion_20261019/234fb9f055132240b232.source': 'd4afd080da34410d966f8ad273b13c110181fe418532059111495d580954625b', 'data/raw/career_expansion_20261019/ad5bcf56aa6e4cb2a7f7.source': '42e22f2104ad2a2954c307fdc0507bb600dc99c0998824ba2d6c620f037638a3', 'data/raw/career_expansion_20261019/8daf994693ad2cf54610.source': '74e8f7154835d8109287adcdab791197a628af885eda41b18ea63b02ad9ae48f', 'data/raw/career_expansion_20261019/8eaabea4bfa48fac5132.source': '4fbb86a27203cfc0bda3d5606b064e663795c7c686595429cbfb03c3445a8a95', 'data/raw/career_expansion_20261019/4dc74d1ac0d4de6601ed.source': 'd631b03a12d1c9ae4c88014b0cb2053b4a5f902767a7f3dfd9f5b575b75bb080', 'data/raw/career_expansion_20261019/300f28ffb18d146a5216.source': '674bd06b21178755171af5888892b865b37fea09e5856d2ace913bd22d52a3b2', 'data/raw/career_expansion_20261019/82a132d875e840e2201a.source': 'e82514771abbd2def22a6f22ad998790f06d4997a78281d0467628888ce37488', 'data/raw/career_expansion_20261019/7785c44ad61e892f03e6.source': '348cb1edc139b6577a4c4d056e7afb4f09a9735e35846725eb7915538c062abf', 'data/raw/career_expansion_20261019/a5e54ede6ffb7740f0db.source': 'd9b6744baba978aeaf41d53d2bb6109d186e5de97a3309a4475de6e448cc1d15', 'data/raw/career_expansion_20261019/29aaf9909b429b30c0c0.source': '2c7fe7f45bc6dd0ffd9ee3ea8419f674ac1de63d5f2dd7037c89457b98fd0cd5', 'data/raw/career_expansion_20261019/83d768afa7bffe2f4860.source': 'fc93f20be3d32b305583f303bf8fe1a9fea86fd00d474529c7a6b157ed6e6f53', 'data/raw/career_expansion_20261019/671b49e24de612101dbc.source': '513e1b05510c3bfad66b6c81d3d0db4f1a962eb529f44d046764d7731f89a013', 'data/raw/career_expansion_20261019/8ed2a71bd8533cc8a159.source': '29bca747006f316eefd14c06b9275442283bc6324aa2a02098ce98abb99b4b2f', 'data/raw/career_expansion_20261019/5a460cbbc1e834a8a381.source': 'c4ec364548732c9bcb307d454768792cbdb39a0ceec0fb199f4d5778cafabedb', 'data/raw/career_expansion_20261019/4c0d7016cdc1000d4d2f.source': '631f17aaccd9b795f180a825b067f1f8c95a87fa8cdbb2a201adf19366c42f84', 'data/raw/career_expansion_20261019/e3f655ee2fb541c2f174.source': 'd7440202fe03830f18f5a84d142e2f783956839848af27fa1c47697d40a90194', 'data/raw/career_expansion_20261019/c45ad5f6cb839326f43a.source': 'acf1201cba5e8c179c1a5ebb2177001ce6659056394c18ce5c81f155f8f56824', 'data/raw/career_expansion_20261019/8b170c870f2170688117.source': '11638f3b53a72b5bf02e1d0603df78dfc25203ca1ac991bcc41c2d00da96cb51', 'data/raw/career_expansion_20261019/ce93856aa87a99e6b0bc.source': '8b3d0c0ed79a3c633c6f431335638d3913f32711fe842828097bb7cd4cb5a505', 'data/raw/career_expansion_20261019/6b6f3ef15af0aff30b65.source': '58a934786b4ff496c9f8e66698813ec5489535a4926eee1932f3f82cf3f863bd', 'data/raw/career_expansion_20261019/7e447b0dacc8626bc164.source': '89dfd53ceea70a6f9bd5eed10d1d36f4c7910a010f2cfe732789429dcf9c44e4', 'data/raw/career_expansion_20261019/a53dcbca2c4ff2928e1d.source': '66f74fe1dfa834f83a1c9b662f2e95c87d52ba8c200dd6b3c98fbeab1b348eb0', 'data/raw/career_expansion_20261019/981344f1f015b386e78c.source': '9b5657e81314e33d0cd5bb7f0b14a6401ec54948d4a0417c6c64ae3efddeea26', 'data/raw/career_expansion_20261019/55237190a38e34715f45.source': 'a5f4431a69ae1dc8e0e11eb2c9122c51abbf5cc7aa6a0d775c49fa9b02061a35', 'data/raw/career_expansion_20261019/b0aba6dfba7afaf8bd4c.source': '46b28bcf53305214615a9bc3b661d9a98efec3913c09182cabc240a76a77eff1', 'data/raw/career_expansion_20261019/98052d917560843a2d42.source': '214ef638b59cde7b317d1f9319f900c51e409d60abbc01541dd0c4c03aff7f41', 'data/raw/career_expansion_20261019/160243f53c2b7273a4ba.source': '107de1b9ea1aeda6449540b5dceff5663730de224755387acdf8e8a7575a190a', 'data/raw/career_expansion_20261019/314e07c9bcce40d45af1.source': 'cd64326262dfc21bdab394ee2cd13b9fd9384423f09d953b8c5eb3d05ad49053', 'data/raw/career_expansion_20261019/5075d1762776f87d58d2.source': 'baedadc5acda74fc307e2af3288890359919c49228b4ecf6f79d5de6b64e3741', 'data/raw/career_expansion_20261019/6f0d2302c6bf56230b30.source': 'a21b8bb2f534389e954311d7433adeb033cb35b6a82f04166af01df07f210d5a', 'data/raw/career_expansion_20261019/7f0e29cacad947171932.source': 'dfcf04b8dbc086a396f365248dc93c6e38ebae64595392978883a155b9ee4a48', 'data/raw/career_expansion_20261019/a4f943371e3491089871.source': '20246bd43db3d5cbf4dbda6c434431ca59861174a5f8249d88160b62644b7e4e', 'data/raw/career_expansion_20261019/e040d32c629a25f3644c.source': 'fd547a6c7349ce65f0474907c42bdd446439a824584c0e4f3bde6f9717bc31c3', 'data/raw/career_expansion_20261019/f024190b37e9afd2a316.source': 'e9753a1c4de91ee3e15062f758a865eb61b2f9dc825919135660d0bc1830dee5', 'data/raw/career_expansion_20261019/c5b7cb83d39a320517aa.source': '3fcafc1a0e26a8153e774832b50b3e8150508195dcae331de8f8e21c9ef1310f', 'data/raw/career_expansion_20261019/2e23bd0b6f246556af4e.source': '2b07cc9b92401025656ffc690c826d20d9449445248e5eacf19dca55f9c3b42a', 'data/raw/career_expansion_20261019/cca982339fc9c7520f67.source': '25f29026d8b847e8f05d93d0006acf6d556d926427edbf9a1e31471ef48f19a4', 'data/raw/career_expansion_20261019/4750e205941e6e46cd34.source': '21066ceb9ca341833184e287165438c77baf3c3d333e9593ea1f028ebea4fa66', 'data/raw/career_expansion_20261019/048b1b95415a7e8a5365.source': '97868079bfccdd9f5764f86754505568ddb5192715a47419b758db1152af8431', 'data/raw/career_expansion_20261019/7acc36bc1a9fded279eb.source': 'b428d310cd813aa33e9c6fecdf080d31891934e854e21a3c300fe735131a715e', 'data/raw/career_expansion_20261019/778a156a380405da62e9.source': '842b8c6546d9d62f2568ed84352457911f5933e1555a4452760c4345f5147058', 'data/raw/career_expansion_20261019/e72270ebbbaed52fd5a6.source': 'bac285e5c0d8f8ec1947fb0687f089e9a8b174ae3ec87d69e825124ea9fa250b', 'data/raw/career_expansion_20261019/ef48941341213c480551.source': 'd947d86041ce275ca7b560e39c084cf10e58d2cdf867acfef543d93d977db94d', 'data/raw/career_expansion_20261019/f1c95537f6c5f86c863e.source': 'd43bcf9f880dc20006dfe0a8f01dc11d792ff4e8302257af9951b23b0048d485', 'data/raw/career_expansion_20261019/6d78165ea70db1034399.source': 'd0214947919eecb278f367a9b6c1f657a8fe3854be291421c92871fb607277e8', 'data/raw/career_expansion_20261019/e5956d97916ba2d7a1c4.source': '418bb98d110d3c1eeada53c5f748628a54cef87a94bf82ec7a629247ef241212', 'data/raw/career_expansion_20261019/159325bf4c902ce7a3e7.source': 'd86e1a2314b99cb4f26607d23169d9375b1b935a593faed3c49b26e50ce8048f', 'data/raw/career_expansion_20261019/b84278706729b32cef83.source': 'fcb92bb46ca6e601c44260537465b2f86b8f82d578e1df9f6bef957e57bd5ed5', 'data/raw/career_expansion_20261019/e21fe70f6c03b3964569.source': '656253478127e257ffa03ce9896dbf5487c27f624c1ebbed9d7ae6d373741b85', 'data/raw/career_expansion_20261019/46bf550324e7ceb65bbb.source': 'ea2dc9329fd9d2032314bf9a497c0148bec192f0340fc67ea06c4955abdfd953', 'data/raw/career_expansion_20261019/cc34ccb8d2b78bbcfc4a.source': '617e40a67d9405d93cdd162646c3fd83dca5d55e8761b1edbd7955a1d5dfe1f4', 'data/raw/career_expansion_20261019/56d4547f426b3499926b.source': '0790d431b46773c821d7e6800bc50c3b45e0ea293a5286100fa5a3fbed8a35f4', 'data/raw/career_expansion_20261019/7101a0dfc0e27b39ccd0.source': 'a59fa8273124d7d620ab9b85ab1abf4ef47617df3e8c29d9110522dc3d0a742b', 'data/raw/career_expansion_20261019/bdac1ac49d31270d67ca.source': 'd5c2ab840b5b69e0d076ef70528d7629408307c3cd3092ac12336ca9ddb9da6a', 'data/raw/career_expansion_20261019/ba2a9f6e7eda1e7834a8.source': '0546eaa4cae9bfde08803470cba91be7926784057c2d5f9fb26923a9849cd991', 'data/raw/career_expansion_20261019/c01180cb9200f1f462ea.source': 'c90d151ab4c0b0d2dbbc9830eacd770e113b5d25e5709d4072e2774893fd0c04', 'data/raw/career_expansion_20261019/3271736fd4520295e18f.source': '8b87c68143653d18a9e96f33374f8eb49ac51d19ce34299636af19efc14d5ed2', 'data/raw/career_expansion_20261019/8012f92ae6bb0a4cb590.source': '11291e7d10ea09c93a57d7911206cc890e249d908a1b47be1cc02ac16ec4cbe2', 'data/raw/career_expansion_20261019/5607e2a325ee33361d8d.source': 'c36d93bae556d3837bcaf2cea7ca2e303ea0b839ad41652b6c866a612bdb227e', 'data/raw/career_expansion_20261019/a2df216e4d3e510646e8.source': '5d70dd928d4acbaea2edc7c007bc6f5fff4dadf235eda5c2b57397a67f9fe430', 'data/raw/career_expansion_20261019/7a67fff8f8e5186f6d2a.source': '8e695b78419d87cd06115f6d66385eec27094088e69b5ec89309a59fbbf5ef5b', 'data/raw/career_expansion_20261019/718cf993a40145279aa3.source': '8f24fa71f20457e6a6c88efe22860478d72a7043488682bfb0acc6c3a2448dd0', 'data/raw/career_expansion_20261019/14bf947d3ac8baf46016.source': '4fef2b6b8832ad220933f31e3d7eff29a1005c7606d67831ab8a3153fa266f06', 'data/raw/career_expansion_20261019/6500b562f68efff93c72.source': '80a265bed528211aa708dcd58f7a95db36eeb7f873c6fe4ddab0b3a1dc0973a4', 'data/raw/career_expansion_20261019/69ff1271557d68ac3d0a.source': 'cd242c9370dee3a4831903527cbab4536a1ec62f07f2e2024c7a7ef9c43d83f7', 'data/raw/career_expansion_20261019/25d66ca37fb1b52d6eaf.source': '8336a02a7027f336d73cd13da5ab3d1435b40f08980ae6778dc009711611efd9', 'data/raw/career_expansion_20261019/573bb7456ae1dbb2e9ad.source': '50fc5a782eed611069f2aba130934659114b599c2761e74fb5f9a23542fa4999', 'data/raw/career_expansion_20261019/d91940c62f1a23723459.source': '7e062023cf05355aabfc5f19d568e11b3701bf8f4e4725e76877fc0ebf712cfb', 'data/raw/career_expansion_20261019/235443f64f177a08cac6.source': 'dfc8b2e6bd2ff645e9974f9bd1599531532db4ad4b77017f656224f9d993a88a', 'data/raw/career_expansion_20261019/c76793d96d12af2afbf5.source': '5c0c1ae6e005c68f782d80cd23500e187e1a265fee192df0a91d10bcf7f3dab9', 'data/raw/career_expansion_20261019/a40054398b7e8f407360.source': 'c1d6ad1b4c89cdcdc5a0808eac2b567a07bfefdb483734bc8decb3185d2fcff6', 'data/raw/career_expansion_20261019/408ce160aad930ce709d.source': '3e56a425b7b55e00433dea7e2ea9b8b01c4ca66ea82ad4934a0736b44ca8bbf5', 'data/raw/career_expansion_20261019/0f6cb8fb7434f0306954.source': 'ff00138a3e8a6e71a7e45b3e90abece355eb924a149cd1ab4aaa4bdab0179aa1', 'data/raw/career_expansion_20261019/1a6dba3303320c5e2cdf.source': 'f8ee96558822d4a93eee6431e96a49cfc206797763ed59bbd525ab91c01b68e5', 'data/raw/career_expansion_20261019/31a6fd8ee03f79caba0c.source': '7498051bdcd1436d270117990895935194baba6b58fea891734f4a83d4a87bf6', 'data/raw/career_expansion_20261019/e3a22c5d3489daf8cda3.source': '6ef3258faceba1559d55f5b3b33bedb3b5c5913ff69785351ffb4e7130fd67fb', 'data/raw/career_expansion_20261019/ee6ed0905d7d22b4b44a.source': '133e476c5d752bf6ac635f31a8a512b139c2b1beb1b07b9b952071762a66c52a', 'data/raw/career_expansion_20261019/ae8adaeaad8ae7afaa36.source': '4cf7c9437b56e9757ba167f8f37a7e8b9375beac6b35b0e4566509fe28c322da', 'data/raw/career_expansion_20261019/6830a304365d27c158bf.source': '5d28994c0af5764eb1d43273b1fa48c4bea150f22b4069a7caa07f92d3a34185', 'data/raw/career_expansion_20261019/84248b5039ee25813c03.source': '712b570b671bb11659b6ceeeaef8418c4e3798166e5c6a1ca7f052c69c16ade0', 'data/raw/career_expansion_20261019/33edbd27c1d6ed495299.source': 'e32d27ca03da5832ab3db88f36f3c352a747b99da43949ab9dbe09c4a0629b18', 'data/raw/career_expansion_20261019/efdf1e20d5e9ad4897f2.source': 'de69d41941a8677cca57d4605a43a7bc32430114dedd357d092bc252f7edbae1', 'data/raw/career_expansion_20261019/a0c247fac78a37f5feb1.source': 'fb3c39272f741afe749ea8a020b1a4cd5e30004dc8772037303745f56995fdfc', 'data/raw/career_expansion_20261019/5c83128701cb1aa1c207.source': 'f5b0dd31e59f135ece8eee4ea90a1aa719bed1104f8d8660fe7b78ce7ced118f', 'data/raw/career_expansion_20261019/4b8f1ea0dd27bf65510b.source': 'a3ec332451baed5bd0a6f73c864755f4bb1c13ce7a490681d46a003949a355d3', 'data/raw/career_expansion_20261019/06ccc7952c712902b5ad.source': 'c2c63b9eec943c43bef79ca9ed8eabbf32b8a7c2fd86eae0578104d31f71fe60', 'data/raw/career_expansion_20261019/df3a9247bf60c1e79911.source': '97cf0c47459fa39410b9702d9c46c8257c79e33f39dd21da81bbea336ebb2f01', 'data/raw/career_expansion_20261019/981dc8d0775083a2ad65.source': 'f3f5751a33016b030804799909b5537154070bd8ec6677a546ebf9ba01bbd82c', 'data/raw/career_expansion_20261019/ae0700e23a1d816e7d55.source': 'c8e485f0d57ee6bded5f73f12ce7c462c7723993d660d09a53fa73c53fd6dbd7', 'data/raw/career_expansion_20261019/bc468b9f6d362224a42b.source': 'ae36045d1105365b878df0f5638c925ed46cb789a2b8d0f51a81b8cc0b050d8a', 'data/raw/career_expansion_20261019/b6e8b58dc77833d48feb.source': '5fa63a2b941f0c1c4d1a655ae75ddf37688dab1412a59258ac2d6b07de49df15', 'data/raw/career_expansion_20261019/7ee11e07c6fd5c8bb87a.source': '1c5cabf4a3e6f33c0b7dc3df56be1ce61572f5f844dca1798448c2a1656ff77f', 'data/raw/career_expansion_20261019/8fb25c97b1da43ae8bd0.source': 'dbd04ba156909aa18bc3e3d8e597a16d8a02a6ec2b13adb66ad1d0695633c019', 'data/raw/career_expansion_20261019/b587a5e46c6e1f6751ab.source': '546bf7a06c22f241135b6d64454d4d79615684ec3cb5f7679f053b59361311de', 'data/raw/career_expansion_20261019/90edbc3b20226a41f3a8.source': 'ccc5959c2ab818c43776497d2c8e631553a2203097dabd8b1932f06063a6b66e', 'data/raw/career_expansion_20261019/2dab365d6e6c249d5795.source': 'f3f0ccb03505961da4ddc75107803a7e30d851ef6ec81bd65b2140d94ed20e39', 'data/raw/career_expansion_20261019/c18ac5c1cf06cfa531d9.source': '5755576f806f25f9e8d0ce174c619b9f9e300ae6863fae7d90c013083a256ebe', 'data/raw/career_expansion_20261019/156ffd645630a03df052.source': '2b93371473d36842865f82c45997d2321bff5f64046fe9715910f6207391acda', 'data/raw/career_expansion_20261019/e71639ced460a7ee7fa2.source': '2eda399b311aebb21cf8848f8a17687297731a077d3a7e487f7e18c41dc91954', 'data/raw/career_expansion_20261019/8a80f028866c45058561.source': '18f5ab4fe6932b2f1726c8cc0382a8abed4e37ee2e9d46fdf8b12acb875f5b7d', 'data/raw/career_expansion_20261019/806d02b2b5eb6d2a2d8b.source': '924243382a1458cbcef4363df51c56c6145c51ccf7c0a6afddad71b651700548', 'data/raw/career_expansion_20261019/64937ab1e916ff2df53f.source': 'a82ac876aea775810ba88270c917de751a00500e0b5af08d504ff6fbcc85b67e', 'data/raw/career_expansion_20261019/6b8915ef8fb902feabfa.source': '6b20ad535e95fba23c662a8db3c4de71ef6d5677e95b9f2b93bcb02df11f67b9', 'data/raw/career_expansion_20261019/2e82265b0fa573ab348a.source': '758657a0c67d21ad9fa77c41cdb26ba3766fb3e05e0ac8f5d3c4af72aec8234d', 'data/raw/career_expansion_20261019/d95b2ab94c21ab18491a.source': '06dc2c195baaa090fc7c32a54cae878df7bb6c7368c8cccd87ea0a1e77f264dc', 'data/raw/career_expansion_20261019/606002eb4a668a9a18ea.source': '6a0a025642a80743ce91058c250cb8e3ae1e724569209f26667bd409624c8108', 'data/raw/career_expansion_20261019/0912e18a119d299531bf.source': 'b9f023b3b2aa30d147b291e3eaaa717987ac5ad5a381bb3ddde304ebaba73784', 'data/raw/career_expansion_20261019/70fdefcef4d0c969ab6d.source': 'ff024035ecc5aacc8b7257045fbe92e1b2ec07dedac9e262ed151e80f151dc2a', 'data/raw/career_expansion_20261019/f58477807dd6ac95eee1.source': 'c8056bbcd4a1563d7531ad00855d846dba99b174de81bcc4848fb5cd5b026d88', 'data/raw/career_expansion_20261019/2b0861c3a2c5f49e357e.source': '9da1ff1992430a759190a73cb8450ddfc9ac20e1463e9aae29eb4a3c8fac65bd', 'data/raw/career_expansion_20261019/3b58769159526a827f98.source': 'eb20c5c064e6e5794d23ebfa4c9691d825102e12bd338eedab92ddfc153bbc42', 'data/raw/career_expansion_20261019/f95ee2859b50d98eebdc.source': '6bf6c69b9f71312d19aa12370ff9564c7b0de1efe73ec037ed895ea6ca4bf7c6', 'data/raw/career_expansion_20261019/97e6c275c2a79eb45576.source': 'd60d99dd2c67c2c47fa2d66d9217d74b417d5f59e563bcc79d349ed9cbda3b83', 'data/raw/career_expansion_20261019/b4a3c81d43d437a3e904.source': '58c7ee0eae6552144bb56d1ef21f8e48f12a4adab25f5f9b8f9900d62b675a1a', 'data/raw/career_expansion_20261019/006e447db59dd066a4f0.source': '981a586617982fe17684a0f389a552ac3c80a82366627251d43b5586a3c28d78', 'data/raw/career_expansion_20261019/151ed73396e41fb9162f.source': 'bf9e5d18df4e6ac2a9d551df9e6327138b8847a6a43ecf8002220caf2fbf0356', 'data/raw/career_expansion_20261019/2983c48c4ecc1b25311f.source': 'a3978c7769e5a0402141f22807902a8ff3c3e752e3bffc536b4f7113456781cb', 'data/raw/career_expansion_20261019/2dfb9474a16b950297d1.source': '469e5c9c0aa5d43e42deab9e9316b4f83e62dfe943a041ec806af72856074c42', 'data/raw/social_followup_round6_20261019/fff69c0c06e8ba50d093.html': '9079d0f4f0094d3e4bda4d2e2a6aae8f4346b6484131b0ee2f962c8b2db1ae19', 'data/raw/career_expansion_20261019/ac5c242b3801d2ecc774.source': '5ffa3efcfcf8898e2271af5b8a044172d6c8701462dd2fbeb57c422c921c43cf', 'data/raw/career_expansion_20261019/7ca08e48ff67899ba074.source': '6ed56df05a0154c03ecdeb1e659b0d8eb4adab224478fdc042d9449e2e6afe47', 'data/raw/mayor_incumbency_20261019/moi_index.html': 'f9f20a5b9b4fea428406dbf78522a06fc0e5d579481ce25a293607bf2bc47906', 'data/raw/mayor_incumbency_20261019/d775b86444063b922227.html': '49c626bc51229c9b9fb8be255605a11ae352e8cadf9526b2d19c83900d384e0a', 'data/raw/mayor_incumbency_20261019/f824ebc7e52abb2e4a54.html': 'a8a50f62399b2619b1dad3f42f2f4899b6979d98c27bc72031b0a0ecb51981b0', 'data/raw/mayor_incumbency_20261019/3f63c28273c889f94862.html': '02b8ad56048757ac882074882300a9472d3b49ed950e5ac9b9e9ba954fe0eef4', 'data/raw/mayor_incumbency_20261019/be060bb813ceb7178746.html': 'e7da0c60d1ad621242ad38f89d8d4c10cdaba75a936cb4c3463541068d2cac76', 'data/raw/mayor_incumbency_20261019/6f03f4272784b2b5abc1.html': '2cfdda01922c16316f708f5ae3fcc2b0957a012079ab1fb729cc6b3736892e3b', 'data/raw/mayor_incumbency_20261019/8cd2dd7543fb1fa88d2f.html': '8740ec7752dfe796934cd465d60db776db9d6396fa4c73182bd06d3e328f8ae5', 'data/raw/mayor_incumbency_20261019/83360adf4566989f9302.html': 'b86a60c2c65c71472a35f58ab6301973aa7f5873f84f5716ea62e11d4b6f4048', 'data/raw/mayor_incumbency_20261019/e2263dbfb2d71e851286.html': 'c3dc0c3394fe219721448677411a094621b3b4e425a6412c23739089ba9de9af', 'data/raw/mayor_incumbency_20261019/fdc0102ea196d639c51a.html': 'd1e56a53abe984e0432a212ae7cd76a5947c9c7425290a38bd916dd5517006ba', 'data/raw/mayor_incumbency_20261019/b24fa2c217256c63b01f.html': '956c2b6e81722eb0474e75b113074cf103e9208999af5d1802ffd76c96f72ecb', 'data/raw/mayor_incumbency_20261019/4f25f3245ff93e0ca7b1.html': '5b12431b0c6dc7196c4becb3d062a2944ad31c8a4dea846eed3a3c34c884f70f', 'data/raw/mayor_incumbency_20261019/e4dcc1ea780e6c02d5ec.html': '8325a26dd5ce8ceeeb464c2c02576e6b3ae6cf6e570917890d095eb4d6c4032e', 'data/raw/mayor_incumbency_20261019/8a45e67812065cc0a1fe.html': 'cb57685d4889028c4d115e8fe06b8007b621a125a9a6f5c6d6f986a7be7d3de5', 'data/raw/mayor_incumbency_20261019/2bf00738dba41e2a5b99.html': '129bddfca4ca08942a2e186692c2dc637ae7cf1af486661a1e0a63b1e464ca70', 'data/raw/mayor_incumbency_20261019/608039e64cff8b84f2d9.html': '7800fdf26f783ae400ed56e58b6b16518e4ca5d1cf8fb86d3107da756b1fcb4a', 'data/raw/mayor_incumbency_20261019/2069317dce4f3386d472.html': 'abccbfd39b44805ca59e411ae129a69c904f5383996ad1cd763f24ec3e0ab497', 'data/raw/mayor_incumbency_20261019/233906baf93cb56cbd3a.html': 'eb1e5353613976a50a57339fe4bbefb80133a08c0eb7f6630d8065b183cf5efb', 'data/raw/mayor_incumbency_20261019/207d98c1ff2ab4618283.html': '6c6967eeb64ddca960cda1ef4ce33514897742af91b80216c2864f211de3bf55', 'data/raw/mayor_incumbency_20261019/aa7a42d7b4f798f07dc0.html': 'f94b5308832b628437cc9c7793c48ef6dfb81cb89dec3b587879ec8c21b2a28d', 'data/raw/mayor_incumbency_20261019/157ad5a9894dde6b8376.html': 'c772f7b24f0d4b9e33945bad36280db829c6a09381a664bc5deacfa74ddf68f1', 'data/raw/mayor_incumbency_20261019/0990e46da06cc03c518c.html': '31c2487affe124044c7ec6341c4f388bc6072057ac0aa2df5622a309844654e4', 'data/raw/mayor_incumbency_20261019/27f688a962564fe3e023.html': '0460058f7be66ae71002b11d317b9a1356b443984ff079fea47d7c42031c8157', 'data/raw/mayor_incumbency_20261019/122b46c316476d145c8e.html': '44c1d7c0d4006a03c81522c3f17ed226cc63e2bed0e554fa9646144e1e5ea1c5', 'data/raw/mayor_incumbency_20261019/c1f90b41ce7184d7d327.html': '6605b7488c2e5fe0421f6845963c205c2129589b8ae854318ba3ff4a69845afe', 'data/raw/mayor_incumbency_20261019/ef32153352779b162060.html': '74d613759cfe0352a61426c95aa8a2964500c3fcb591b2ae41d55689accb19d6', 'data/raw/mayor_incumbency_20261019/02a5c146d9fd23d7e268.html': '395b2052c95ec1d48569b6bb53b3b21ce7b5a53dd7f0e6608ba47284cf5a833f', 'data/raw/mayor_incumbency_20261019/ede27d373c1df21e4082.html': 'ce5de43c92954af5e8cdb8255059ebeef4e6c22ee5194b88e268a4ea70e6f5f2', 'data/raw/mayor_incumbency_20261019/40003936c8ad0e5cddff.html': 'e83bf553fa47e528981647007f285ea1d739a9644b7d60bdcf7715df9032f8b6', 'data/raw/mayor_incumbency_20261019/1c2c10475f30177c67f4.html': '88f8c69410a763783d82e2a627ad45da9f2a34fc091ed785c2320e8fd15ae67e', 'data/raw/mayor_incumbency_20261019/073b4b373ff6173a4278.html': '8c76981f0eb952ebea93e09e89eb0feb2b35e5130ac918535271ae5966a16eef', 'data/raw/mayor_incumbency_20261019/75cd9f9fe7f13bf99700.html': '709d2f8a4f4587e95c7a4230c09084b714d1100cc1df7bd21af1d6e244816b86', 'data/raw/mayor_incumbency_20261019/cfe8b305f2071b592728.html': '45a68b1f6b8928fc272005967f480821df70aa8edeb2307b9e39049b808a4f54', 'data/raw/mayor_incumbency_20261019/5c923841133f53b0dd00.html': '7d4677ec8995b67a1857d4f8eeaf0d1dea2ffedace6a976c2cd8a6340e342f64', 'data/raw/mayor_incumbency_20261019/0f3e74096173a98f900a.html': '58d1148fcd00df8c3f9ff93850e45422e492fe38657eb1b32e674e0b3137fd68', 'data/raw/mayor_incumbency_20261019/8744a445376c2c429df4.html': '9cdc76bfef9fecad26f423153cae4df9beae2c05f50fa0f860e49603e4a85d66', 'data/raw/mayor_incumbency_20261019/e51321c3a170cb17abcf.html': '0ec82b83428028a06f298199fd73ef52608281f78a370025668273d54886b05c', 'data/raw/mayor_incumbency_20261019/85b772a8ea1eae944340.html': '13b39a808b0b33a1bb6890273d300f8a03113018b711288942a0988879dc2160', 'data/raw/mayor_incumbency_20261019/4212f7766527fc7a0b5e.html': '7f066b480b576861df30c2c41ffd811f1e2d503527d132909725762db0caa1f7', 'data/raw/mayor_incumbency_20261019/a17fe3e388acfb4b7522.html': '29f38c1abd640b0eaeaa4984404ae3d2ce9ea9ed94ef947aaa6bcbf797d6fa78', 'data/raw/mayor_incumbency_20261019/454b4e8e4da6768fe0b3.html': '2f21a7f64b70fb9f158ae7e11d055e3b7063c8008b5e3571e4defc8ae5fb0741', 'data/raw/mayor_incumbency_20261019/a6dd97f4cf6b64a9f68b.html': '1e8c12b96b877e8363b67d238d164aae097cf48ce12ba144fd42f4ac36810a48', 'data/raw/mayor_incumbency_20261019/5cdbe4c6b92575425c58.html': 'f0c4cd788f4c76d8de77ecc41cbede3d12db7ec714723191898d0c4d872e3f3b', 'data/raw/mayor_incumbency_20261019/24f373e67857b984a4e0.html': 'e97ab18eb0d4aefcbd78db0e0bccc3a109bcd85c78be2a901b1a9b0693a4164d', 'data/raw/mayor_incumbency_20261019/d62605112e45c0e1c66f.html': '07361275985d152ceae262c5c174176ccaff5a977f1364098f94ec5bec8a2f73', 'data/raw/mayor_incumbency_20261019/9e659e19edd027e0d2cd.html': '0df55f260967f2fb89f2ecd1ad4b2de4a883c9e6e691725120d4c2f088bf3fbb', 'data/raw/mayor_incumbency_20261019/54f41a2a2a3f085b2b9c.html': 'c14a7e36eb9e3816d599f37c13d68437bf8698e3c3a922757ce1a503ae14c231', 'data/raw/mayor_incumbency_20261019/d566003a726a9caf68d0.html': '4cb8f80108204cd7168544d2ff9cef78b93319b8847fc9ec28c1d7f99af281a2', 'data/raw/mayor_incumbency_20261019/90ced26583c9f6adcb71.html': '4b550a284d7095a49c5fe56da0cfa856b0145b4975c3fedb35c47db00303df8c', 'data/raw/mayor_incumbency_20261019/deea521c419197efb7ec.html': '79469eb6186b2ad23073f9aa9e8563052ea51f7d0455d825e45bf83f5b91d55f', 'data/raw/mayor_incumbency_20261019/c595659f415615648dcc.html': '8cbdfdff70763998eec2ac22b32ea00091eaa12058701ea89b98c73e84495a7c', 'data/raw/mayor_incumbency_20261019/7cff82b0f5be32e7cff4.html': '162581fc0acfbccd3e3085e1f09ff7ea1be15287bdf041557f513f67d7caa97b', 'data/raw/mayor_incumbency_20261019/70174b78ed1026a925cb.html': '2b903d21ad6c7034dee0b7a6a053af81fb4902c81a6a7f78d1fbae042d660498', 'data/raw/mayor_incumbency_20261019/30f55321c8f179fe98ef.html': 'ae897455a32cb810029cd4fdbde78e8ffa052831fb142889323b01fb0a151e5f', 'data/raw/mayor_incumbency_20261019/82e4adc2a3461664a2bb.html': '91c942ce5b283351c24a527c4c5e276788356c38bbd28640d1054ee02d5fb1da', 'data/raw/mayor_incumbency_20261019/a363a8be4dbf3de4bae0.html': '5803b490dc7c0ace00f4b7403649e8db238e78d83777afb4f2095f3f8e59e615', 'data/raw/mayor_incumbency_20261019/66f6e23e1c45cf922ccb.html': 'c239fd1e238bad0dc763e3d93e8535aa5fbe665753b5f875f8294fa70e37f31e', 'data/raw/mayor_incumbency_20261019/0b983be118abe6ac6719.html': '05bbf5f3c8c7d3d5d2555e9d894f1be86ec2e55299fcf8339ff43a3d4d6cbbee', 'data/raw/mayor_incumbency_20261019/a18da20a790a7a402c1b.html': '45cbd3e03d4b361270037bc114b970a9d4d96b582db8e6f28fb362150c5deeb4', 'data/raw/background_expansion_20261019/8892bddf0b1b0306bd0c.source': 'a1b1c51b339fe286b15f44d0c512db8540c0860393dc1798f237972d92533fd3', 'data/raw/background_expansion_20261019/0cd62bab8d308a0a704d.source': '8706be0d94b0e7aa125074c48dd9dae5b4bedd214aa11c04566a7190e45a72d0', 'data/raw/background_expansion_20261019/9daf94d7a1c2ce39d031.source': '6a367b4e6fb63d2b260b46831a6a5decfc6c1589732e16f3c8cb572ab84eba87', 'data/raw/background_expansion_20261019/681b86a381c9e7f59dac.source': 'ac98d31efc712ebb36cc0f11956a6e94cabfdbcf5b6be637b4ae44605cf88b21', 'data/raw/background_expansion_20261019/821d9d4d33e0a20562e1.source': '6499e590ba0252be4749c3fb727fa483be640b23f53bec67828ef5d8de9938e7', 'data/raw/background_expansion_20261019/9ba10515451ba06bd018.source': '10ba4682de4d75129372617fe255ea6b1796666c5cc19219b5d236ec64cb4538', 'data/raw/background_expansion_20261019/53e6e05bcfc46eb1d497.source': 'c2cc1bc5458d803685f8183c630f133632f8e31aa6403ded2c191ccd9471fd13', 'data/raw/background_expansion_20261019/5501ea4e61759c8737e9.source': '9d9494d89b39ebec4b3991662ef033305a188df26123a58e87c20554a8e6423d', 'data/raw/background_expansion_20261019/40551a0736b2a062d95d.source': 'bad9a972e95572580070c4d206c208a8d2dd413a117337f24f9d160fb5dd7cba', 'data/raw/background_expansion_20261019/8a931f7bc6ba52c2de27.source': '98a4d7987406e7fa51db1e1285f9e62c4798fe7eabd4ca91dad6b102262d5ea4', 'data/raw/background_expansion_20261019/3779b8d372c1fed96c40.source': 'cc9b33e197784296a4c5d2f4a877f4e49c9add8d14108e84aec2151d2694fff8', 'data/raw/background_expansion_20261019/04d34aeeacf63870bd46.source': '0186caeb942422088287c8999468b7ef32771ee582f7c4a6b50e02f8cb291039', 'data/raw/background_expansion_20261019/be73514bd169fa3e1ab5.source': 'ffa8b3c1e3d1fa07adfd783ef64fedac0f7639d3f1e170195cb58b4b1ca736a5', 'data/raw/background_expansion_20261019/8687d0bb7b705d893ba3.source': 'a51d2742fe0b3b047e1a6dba97eaaa21df44819587ee01c13ab55e7a963010ab', 'data/raw/background_expansion_20261019/8b852a13e544190d615f.source': 'e0f28b71bde3fe6e777d0ec983ecbe735c531341086ef80fca5fb5e8185ad9df', 'data/raw/background_expansion_20261019/a22eefc4a2451ff54ec1.source': 'cb1e555625566d45137598627a7cbadfab65d098d21d1b53d167b857763ecd9c', 'data/raw/background_expansion_20261019/bb4ceec156de6753086a.source': '4e77f34110fcbd55bf6283a6e078c0e81e6040c0887b00601487f52b78510f6a', 'data/raw/background_expansion_20261019/da79f2c274cbd8b52312.source': '15bf686e728d9e8f4511932f42e0e5520a1486557866d94873fe4ca9e07727a9', 'data/raw/background_expansion_20261019/da833a5296e4ee0c0de8.source': '60e9846e0d1d9a9678151bbaa7c214bcc3b092eae0e3fa61d310153ff6579337', 'data/raw/background_expansion_20261019/a5f34142ce1c102d0089.source': 'fd478762711a58d7f96988e0e3a6ad983401ed373d700b838e7fcd291c68a160', 'data/raw/background_expansion_20261019/94aba999f4ca969c65ed.source': '7b8aeef5e279d35e3da10d2a46c04a442e634506741ac5b19e5f06aeeeca8212', 'data/raw/background_expansion_20261019/48f94a8fea75d03f32a3.source': '7b10f9bf6d2e11c680db9cfb3608b73edcaa325d68b7560756ddce35233fcfe6', 'data/raw/background_expansion_20261019/c61bfdd11e0e8e1ba8ac.source': '68fe4ebceaef87e6ce3e6874c8b525e23969c1b9abcf7e8518d4f48b7777fbdf', 'data/raw/background_expansion_20261019/d7104b9d543701e5eae3.source': '16311b8917ae90795db55be15fb97cf8a8742aa18d3d6e4ba3b75c7a7a44b09b', 'data/raw/background_expansion_20261019/64b85c4b76d643120827.source': '98aa8b1a6ef7b6b93674238b4e4bfa7ac1a7aed96323e4aa1eff0be89b38eebb', 'data/raw/background_expansion_20261019/ed5d2b937362694f1ccf.source': 'db6a71eba60c502f497f16445645f5686c8ae61f2fbf31b7004ce4acd80d5c5e', 'data/raw/background_expansion_20261019/d9f80e0c91a58a9e6658.source': '916c8f52e7e1901418836a498701345f057ec0a5abef661ec6d2384a6781fe87', 'data/raw/background_expansion_20261019/b8f7e32decbe1d442a10.source': '9758509114bf2bde7cec77912f6334b1d186df0663cd85292b1c7322d83c500b', 'data/raw/background_expansion_20261019/e04304f484d197adf0ed.source': '72a86ec5e4047437b6f97983a77617c0d2ba7646503b37dd0a7723e6e6873537', 'data/raw/background_expansion_20261019/2aa85add8b63289c61c5.source': 'd37c4febb720eb57c970851babefd1b3ab8fc207123b23a01a94c257f1b33ec8', 'data/raw/background_expansion_20261019/a9ce3333d4f149e51e7e.source': '77f915a3b667626d691200f329c398c717892a6f102567cd2730bcdf6704bcf4', 'data/raw/background_expansion_20261019/50635a884a83577d36c2.source': '13d7735763029a9e8026c9cf79aab841310936d0024799af09f05969c221fb1b', 'data/raw/background_expansion_20261019/6088f01cbb6354da9f2a.source': '47da717bececbd49017cdcdd7057c64ca3f33469877947f00c4632b94216ce18', 'data/raw/background_expansion_20261019/0bf134564e19a823d3a9.source': 'f5b38bc47b1be7256b1ab0cfe89f6264fd91de120b21514c63aa503e53fc995d', 'data/raw/background_expansion_20261019/a4cc4bf704e91a3d9120.source': '66ffdad22db38f838e41c03a11fa3c09bb7d2bfc339503fb6cb1e6a256ada3d8', 'data/raw/background_expansion_20261019/5494d621794fa701fdfd.source': 'c2a6f79686e2ae33d10e56ac48d8cfab758883da48a52112d666e370a770278d', 'data/raw/background_expansion_20261019/89492278723579c3545a.source': '404a771222b930daeecfc6aa4eea0fdd3ed50e7bb56588cb7a726303e9b18931', 'data/raw/background_expansion_20261019/30b476badbb71cbb68c3.source': '4b266a1c6cf581c2d883bf3171d5c6780f23c7b83b150e38de777157027d2e22', 'data/raw/background_expansion_20261019/577f595085f01dfad7cd.source': 'f1add11acaa067b5f99c7ed5543981eaacfd15fd10875ae0e6f0cbbd6c312d41', 'data/raw/background_expansion_20261019/fd4116248c7d601e7989.source': '77f62e06354362f149f9b73fddfaedd882bab2afcc4e53fe8459b4c2008c5f59', 'data/raw/background_expansion_20261019/9d98790d52bb2eca4e25.source': '107240d61385d646da46447858c4b5c93657ef3b9e6d14f851f341a74631c399', 'data/raw/background_expansion_20261019/5c31faadb29422aa45d6.source': '7b6bd7afa421fe8d71e8cf63646fe0520ad20c3c0e7d59c8e6d15e8fb63d67d3', 'data/raw/background_expansion_20261019/7f8b8133411ee4c378ef.source': '6a8b75112e3614bbba54f485e41c610b2ab36b7603d9a9ca394e7b8f6835a78c', 'data/raw/background_expansion_20261019/74acb0084d1321fb1af0.source': 'e1fac1e6a1ae5c90a3c7d2b03fd428ac90cf6ba4e5e887fc7c14c3ae3e9f77de', 'data/raw/background_expansion_20261019/295bff9e889077e8ced9.source': '8d3931074a6171426c242bcf61d1e35a80b10dedb7259467504bab1464722518', 'data/raw/background_expansion_20261019/eac7c3986c4d5218222e.source': '7b6ba15ddda81d8281097380b9682e53c512b4bd40c9219dda71df111fd4373a', 'data/raw/background_expansion_20261019/e67e3059a6a7506db76c.source': 'd3396af7c81c3b778168bcf1e66cd119c155dd1db61b101d3bbbfd3b714b375a', 'data/raw/background_expansion_20261019/3d9244ad59dfbd3fc102.source': 'c02e3d0dca8200ef9f10e1435d7eb4d2a00cd9f39f7c552ec53ea5763633e1db', 'data/raw/background_expansion_20261019/4e31033e4b4dfb0bd917.source': '499bc790d398b320784987a174474d275424ec1c155c555993c00ff654f3807e', 'data/raw/background_expansion_20261019/fdcd8534efa385e7a149.source': '156af849cb0273a732a17fdab48c25fb7b63a772fcef4489f5c67a02f282595f', 'data/raw/background_expansion_20261019/c9785911ed1dbc03b520.source': '21d5110d213574f74e29615e544f5b461f1e8e6a83605f9e7df51f8a399a6e04', 'data/raw/background_expansion_20261019/82122d32aae7f917d5d9.source': 'e2e0cf9fb76cd3c372ff008805fd8ee94936db304e98578b9fb65698ac291b65', 'data/raw/background_expansion_20261019/db9bd9790bf4105933bb.source': '3fd80ba83ea6d0dd455e19adac53016d8327216c519844623784dcc85cbd2615', 'data/raw/background_expansion_20261019/2598e1568ea0239782c3.source': '478020968e97ccf930b566bd36eae716147f08ef8934a63d28a01c1de70456a4', 'data/raw/background_expansion_20261019/b45a4362cd713d39b582.source': '22869ab3fbc6993925a44515a0c149e16f2fdd544aba68193559a69e0e983a5e', 'data/raw/background_expansion_20261019/b075c69157a9e53f0cea.source': '977a118f9f024718e8c997d47d6980af69c87acd2b2bc775ce682525f4a07167', 'data/raw/background_expansion_20261019/acdab1d68e141bfe3ba4.source': '38e4213d946732643ba189225a431c0643391dc7ac60cfe0bfaa7ab1b909eb55', 'data/raw/background_expansion_20261019/71ee84068a4533565285.source': 'cb105f02b7be832a1928840f62096f90fba8dd785431c3d0354272c4fc2df64f', 'data/raw/background_expansion_20261019/c9d060d93e52b6570d8e.source': '3f9957cc4c74519d49feaaf202279b7b539de4148fb57458757826250c50a16a', 'data/raw/background_expansion_20261019/8be785d16a216d7de892.source': 'a02a1bc997e6782d890270108881befc2850d4dc897b9bf4d12c61839fcf358d', 'data/raw/background_expansion_20261019/c8949cc45fbde8764670.source': '565f04e36e9a13ed0dca44e0e7596558daf25d6ed887ff0bb0a186c23283f525', 'data/raw/background_expansion_20261019/4e8422a978f4af794806.source': 'd0dbe33c48a4baff80544b80659964e37bfb67105992ca18c59c94da75c3905f', 'data/raw/background_expansion_20261019/6cb30646d6b2612052a6.source': 'a0e0f59d7b6b75ff230ce4b8b7dcb5090010322f41ea6439b58dd25e19ff75b4', 'data/raw/background_expansion_20261019/d32be3fd8133400e8563.source': 'a895219faa6b4285ea77a55b29a136339fe91c19028723dc4c0ee0dbe751e85b', 'data/raw/background_expansion_20261019/c71bad6b24c9be8cf52e.source': '7d3e51b0eef4cbac798ff945e918ba24c979fbc7a20b767f7011c452efc117b9', 'data/raw/background_expansion_20261019/c566547192e639e3fa88.source': '68dad95c3f2e71dac0754b1c12a361787c542b07fb76f20f04337a827712a0e1', 'data/raw/background_expansion_20261019/9073f918ed631c19e489.source': 'bdd0c24d11e9551700aea98d8bf44c32e60ba3241458af2d36072967ece4ee61', 'data/raw/background_expansion_20261019/ced0ecc46a48dae115f3.source': 'c4f576c27ced3eab7b4b84ab9e81f34520d7828c756147d9d3ac3bd635c2d68c', 'data/raw/background_expansion_20261019/d3287b7d703463e39f2f.source': 'eecc89c9444e653c7151beac509d4b9dac7a3a562129bc7ff376b128978838e5', 'data/raw/background_expansion_20261019/bde14106a352d624d0ed.source': '57ccd71747ad7cc9caa23a35f95be2bc52442aa862330260b2e7e09490e63b54', 'data/raw/background_expansion_20261019/1d39d2b3a7e1f74d5601.source': 'bb4515091373e19150b890d04aa1c31357034da82f0a0dd1c5907821b8386782', 'data/raw/background_expansion_20261019/3660305616d51d196252.source': '2bf5527a4a6a69c54d04d6fd4c919af1792a48bb49cea7ed5d06f5714c766459', 'data/raw/background_expansion_20261019/8828b2c22096bc1f7bb9.source': '7a1dc0182d2e12aae1f2d14ceb7cb82ae45759bd3d4b51d484d52f55f7a47da1', 'data/raw/background_expansion_20261019/3dbf11175bbc9ec34340.source': '05294c8f0cfecdad5cb89b2bf17ee3da8cd67694c644575e8c71cfa6f71564d8', 'data/raw/background_expansion_20261019/39f09ad4731210be1054.source': '22f53feae4fdf0a98207bf514eec14e79d1ecc0d71f9a35541fd5f09736e6b69', 'data/raw/background_expansion_20261019/170afcecc84daeba8b74.source': 'bd0a5d3e901da00ef5af6d69344042f9a2e75cbd001f0c314be76b9544099c8f', 'data/raw/background_expansion_20261019/f4e31beed5a2aad978f4.source': '630d228f9d56b70d2c2775e9d39cc9c8c4f10e56b5868ceea7b96146213640de', 'data/raw/background_expansion_20261019/5c0045a6185708c82e8d.source': '1ddb1e284bd6e6e59671838a51fa1d3fc05e8db2aaf663381d5005791ca6dbf3', 'data/raw/background_expansion_20261019/d50839cc8d4826f5fc3d.source': '6482e7938ed54b45cdaab0591d774071d8674165a9940ddddba7b31a0d6f46e2', 'data/raw/background_expansion_20261019/5012666380716e1dfd3e.source': '9a73598209ac532cc8af1dca1577b3100900c450b466be637e9c48c45f95c227', 'data/raw/background_expansion_20261019/cf723046fcf3d4774d65.source': '3ed339d7769b09842736b28c99828e42be5f48825d56f65cd80e2ba61e10f024', 'data/raw/background_expansion_20261019/90670a66842119f4b61b.source': '60f1fabf0634745cf036863bcb2581d34368b8484b96ea4cd75ed7c098a5d25f', 'data/raw/background_expansion_20261019/d650885062f90165ffdb.source': 'c9c309662aa9aa30e9827859a8696df3bbc206de7546cbdebf809398e1b37111', 'data/raw/background_expansion_20261019/074b80afedad88a8ed91.source': 'ba590f32dfc3a898d787d9afc620d4907e49b9a1ff0b90c8b0989b161a0e1106', 'data/raw/background_expansion_20261019/743bec441d61fd996a36.source': '5dcc2189c57e82ca7a1d2251aab8e0fece91f14c168b92af645b1d84cb8919d0', 'data/raw/background_expansion_20261019/0aa1b1efe052d76be89e.source': '5872f779b483b686db2dafc59136142d8ef171dff16413860fd9edee13a20647', 'data/raw/background_expansion_20261019/b186cbce05ff6a2d59a8.source': '2457c538c59c2218cee743cb81def714e7f0aef331387a55e109a540b125c9c5', 'data/raw/background_expansion_20261019/cdbd4fde0f3e45a2eda1.source': '83b281bfbd230f7f8531e62b3081e474d733c023f3549ccf78be79fc9c329dfe', 'data/raw/background_expansion_20261019/b4920d1821564bb3dee3.source': '4563f2b842056310792b8df6a115c29f0ee05bd634b2f045a14a4cbcee642286', 'data/raw/background_expansion_20261019/2426d30390f413f87645.source': 'ee8a2e28b8d545d819c18fb3a5120ea85e2012bfa0c3bf790a7f593c1e98c8e1', 'data/raw/background_expansion_20261019/5434c68d08791692edfc.source': '9dc09f4e8a3fe51d7638cf4ecb56716b66ea634a7449a34bd965813da1dcd951', 'data/raw/background_expansion_20261019/ba19776149d6499e1fcd.source': '005e51421a968433dcbcaa5bb7c41b255517c479b9f918b9923d0ae57836c317', 'data/raw/background_expansion_20261019/e76ca91dbc1031b7fc5c.source': 'abb004c106ee953d32e515d175ca6aa419e38f45fed49c3d57d4657c19672255', 'data/raw/background_expansion_20261019/0f42bc6d54530db50457.source': '900539903a4ce278e2baef798a8fbc4092945d4a834ffe4f1802d11a06cd1c10', 'data/raw/background_expansion_20261019/8306170fb69203977b81.source': '4a91389bd4f38d2a4e79cba3934f6c04b49eb49066c6ef6b519c2b26a3c69c9f', 'data/raw/background_expansion_20261019/a60063c5d0744c577760.source': '56dfba7c76afd9f2c908a405a07abaf4fde85727bff33388162e38578f1713ba', 'data/raw/background_expansion_20261019/39c2d5f6982f6a4aa5e1.source': '1906b052d7939b4b05d0a78adec02eeb20af2ce0e5e1f2d232e041b4b042ec27', 'data/raw/background_expansion_20261019/9772611b4f6d54787c44.source': '65c0c87a6e643bfecfe2b065ddb91b57d0e87b5abb8291618accd3761f692f9d', 'data/raw/background_expansion_20261019/63cc2e0a1b78e849450e.source': '5410e8a3ba596619a27b5f02bd47f10474531695ffe06008f02b307306a7ab4c', 'data/raw/background_expansion_20261019/976c00f5efb22c54701b.source': '26b9bf6e9261a6130d65efeb47240dd61e0d4f75e7d544415a8c1cb85ffb3836', 'data/raw/background_expansion_20261019/6e8f1cc150a52904fe36.source': 'cc857266dfe2b8de27d4adb2f2af775949efdcb2577e2ead8d4e1a629fba0a3b', 'data/raw/background_expansion_20261019/3e03e83f77a2b243c085.source': 'f81c77875652208d2e4a47e56488dd1683a6f65953a3d1f181514bf0dca2653d', 'data/raw/background_expansion_20261019/7bf9a952ebcd64401212.source': '96373d4e8bb8119e49a78c13c74f0dafecdf054121b1a9eb988b4487f7fc403f', 'data/raw/background_expansion_20261019/87e8a4fb8e4ab30923c0.source': '5229da8e68e7c88d4886d1439e7beba6dde7892cc8111caa94d4ab886b76541d', 'data/raw/background_expansion_20261019/62c44b16f585f2a77a70.source': '6ed64d8fbd218f5e9d65303ba1864606bfb0c04a38eef9e2f897917edc344c38', 'data/raw/background_expansion_20261019/aee3c94631f35669d3ac.source': '5c6100eb2a358db342b8bde30a2a681bd156d011d5b8057f3227c7e4a4a0df23', 'data/raw/background_expansion_20261019/9a4847c4a691a5b7d03a.source': 'caa9b92e7e452bab385099afc2535bc90a1bc9b202a664ca60e3f00e75a35990', 'data/raw/background_expansion_20261019/b2fb7d508b856cbda059.source': '6859385f7f0a8506635c4af3af5c4ad114b7defc808b52c02dd083901900f182', 'data/raw/background_expansion_20261019/5b0285ce8dc085db7949.source': '6f7f0e6ac86b5719b8a27fc0cae839ea47bd14657df3dd3be74f339114bb38d1', 'data/raw/background_expansion_20261019/ec3bc58c428a031f1c21.source': 'da1867edf674a247df5fc8a73e4a223e629dd415fbac2d271858f2067c4bac22', 'data/raw/background_expansion_20261019/7da45840ae69d576bf34.source': '11281469c17f965a957d21ea2451f92a7c5bbd428ed21db75fcc99736cac681d', 'data/raw/background_expansion_20261019/d5451bada0b596585215.source': '0d5071c61fe3d2da44bae817704c25aa5480f0cef58e10412c4d16891479fee8', 'data/raw/background_expansion_20261019/93bb54bec76ebd3bcd09.source': '7a2f448db6f9a4b229c402352d19cd762c6ca701325fc297bb0dea2884b4d991', 'data/raw/background_expansion_20261019/c5c7dbfab84c252ffe6a.source': '733712aa656bc8c1d5be99a1d4c3e047f327fb177064b42821311f71f93dc0b4', 'data/raw/background_expansion_20261019/0fa9533d4406e7dc1618.source': 'a37e937338c2ab3b844e53c70657c787f55d5832f6f898a05cd1b1ae8a7e2b0c', 'data/raw/background_expansion_20261019/531273496ae5d1a7061a.source': 'aff519fb0817c7c9418b6fedafdb1d8a3041ce061b380b9f40c90a90d0b3aeea', 'data/raw/background_expansion_20261019/0b7858b52a649762de42.source': '72b318a7902c52e15c0d06ff160279efde15120564a1bf72a4d0f36a6fcf466f', 'data/raw/background_expansion_20261019/e4cdb31db6cc8037550e.source': '848afabc89e51862c8d4323a8c6a35770f31acea567f76b507c0a5144947b60a', 'data/raw/background_expansion_20261019/f98ab2fee28bd2a60b94.source': '5770eafe339152bc0c69591556c6189372f7ff18c4b917e7496544e71da75ce6', 'data/raw/background_expansion_20261019/02e0facb3e5f4a1b96ef.source': '150ee0cf93d5a870ca22da8c6af3ee5e460d0fdb27916252cf0a52d1cfd9cf89', 'data/raw/background_expansion_20261019/6096e214b1aa177a0d72.source': 'fc595d77af06dfb0db891fc6739332414242c08b9410b50246dacce07b682a27', 'data/raw/background_expansion_20261019/fecd3af4ebd01ab77075.source': '79c5b12b533387ef76fb5a1da04531c24ba15a5cc3737d4213e4a75d3123be14', 'data/raw/background_expansion_20261019/0792158f8ffd55db3d39.source': '7044a436333f88c6ead8b1f1a735a3d109dd8eb04b7ea1758cb2b9ee4e67b103', 'data/raw/background_expansion_20261019/df51f0cb1a001086ff67.source': '0a19b9d86f625e8661d26f3b67abaab795c483e33612fb8866f4c3d8ee7a3c01', 'data/raw/background_expansion_20261019/27d7f9429b850eb84bf0.source': 'afc6af7d6d3c765b20dc7cff6fb8d2fb0fbfb30b4c6d5dc01c1b9c429a1a66a4', 'data/raw/background_expansion_20261019/28f612c793415a78b415.source': 'f24df3876be360c0d9b01b56a76b2b72372bc8cd671f2e469bff39cd59263fb1', 'data/raw/background_expansion_20261019/4a91729c0b3e2cf9be41.source': '6f9b49feea9d2b7bbe6938bc9462cd8d167d63a7b488ca65f334cc12192588a5', 'data/raw/background_expansion_20261019/3d64c5d585f8834f3fc8.source': '76530afc315f176589f7bcb9d640e4c8d5507a1fe2efab00b353f1c77eaf06f2', 'data/raw/background_expansion_20261019/8e68f661ae741e6d4b84.source': '5804298228146b996d7ce0dd261d6e667487c47c58026b2da590aeed1b8ad15e', 'data/raw/background_expansion_20261019/2eb4fc26b1658653119d.source': '46041ab36e4b335140c3c1ca9776f93497198a8abdf2a6f845ffa0aece34399e', 'data/raw/background_expansion_20261019/6910dd556400166b16f1.source': 'e000d17ce983036a8b540b7e920dccaa5df60f6c6c1e73203a71598aab13a3a0', 'data/raw/background_expansion_20261019/d9b8f363d58631871948.source': '4d57a59d40396894de63c7615ddf0c8b864bf962c9f6a9478db9c9d388f07ce1', 'data/raw/background_expansion_20261019/2a0e83266c8c6fa4a890.source': '4eac3576dfc9e9e48705fe66ba086fa6c20a38b9669a6a2bda0dfc47159b9c97', 'data/raw/background_expansion_20261019/2dfc4fda71de93024e05.source': '90ae7b2fd76724f91ae0c626346f9943e67da23ff15bf4a2a1c59f03a5b3670d', 'data/raw/background_expansion_20261019/d1d48715956bbe280db6.source': 'd21d024c8b8113508cb959bd039945d62f86e5b7cd5454cbb66b930417bd116f', 'data/raw/background_expansion_20261019/48f6280c1fb21245534b.source': '9610a5c1db09bc29ab7ebd3fba828f97f2d33e4e0cbe768d426c5f1c05a99bed', 'data/raw/background_expansion_20261019/1bb68d10ca9621c89d46.source': 'b8f6d0c022802641b7ba2d9c94926ce6248b12b7f1fe4f2f91d2cab834ebe98d', 'data/raw/background_expansion_20261019/34a9c6bfa21b830c6fd7.source': '9f05a3c0a8f12ec0ec7905d6d93b592edd72da595939e42efb896a3677a45e19', 'data/raw/background_expansion_20261019/b0d2e772f03f70dbf959.source': '295241899180ed7ec9ac78cce4ebd47dd167efcfd29353ca2b92cd80cf241827', 'data/raw/background_expansion_20261019/ee66a000f706dd0c599a.source': '805884489be7ea0cd70c97172205747bc317ef0161c907d64ad4ef4e62d4bd7b', 'data/raw/background_expansion_20261019/fa2b3854bdeb8783be0d.source': '4a04891e37a5effff04f35aae3a5e7c38497e2e478b680bf4a3b81b200fd3942', 'data/raw/background_expansion_20261019/36cf12074261ab56832b.source': '5f041b70f3e89e4f68fb7652a0d6875d14e9fa72603e0b16524f1b30b716967d', 'data/raw/background_expansion_20261019/eb3bc65501419a51bc83.source': 'c948e5b79ece6b7c7897cbd371ee65a913addfd8cff20e84a6c292060c11ba72', 'data/raw/background_expansion_20261019/f35d8a23acb9d61b4b60.source': '3728ce2eb9c908b6868e08ad5279dc41700e8c18440ce0128ca12d5b7192decc', 'data/raw/background_expansion_20261019/18820697dd23dcd30245.source': 'f1bf58a32c0325d7a837c6d3e6a4d5280eb423ade43a381592399618e38cbd86', 'data/raw/background_expansion_20261019/d6e845fdd56b953e2b33.source': '81a785d23ac4d2acc298f4c987e09886723b9c095d53e91e87ade6c6c11f9e50', 'data/raw/background_expansion_20261019/784802e9f5b67c1baa50.source': 'fc51c5e6b301aa1e2d10a652f9ce3ff8e4f7fff5d3d7858a6a515c01aecdd47c', 'data/raw/background_expansion_20261019/1cc42ec60da2b3d9c1ba.source': 'ed7b03a581c9da77ea5504ee4ed47343a314a01b2d73ebe40e3474b96c19e24c', 'data/raw/background_expansion_20261019/80e5effb96b863d08db9.source': '946fb8e3b07b3f632eb5a2109f6c43dcd9c386e250a13367745d5c8dac88a476', 'data/raw/background_expansion_20261019/5f650dd3919e5bc81f01.source': 'de63146aad7bb07e9a78c26285a69b7c0476a98d7112a19c19d710a1e2cb8362', 'data/raw/background_expansion_20261019/3fe405b88e8c0c9ffedd.source': '70a79011356d8fb432a5ea1a86f67db022c876bb1843bd4d1f315cd59c78a3f3', 'data/raw/background_expansion_20261019/051b9f6cb69d195e8f17.source': '8c7f01549f18d5d81761f807216cf43dc38372e50cc5f25e513288c79b0907ac', 'data/raw/background_expansion_20261019/9a4c3b69f4711d3aced1.source': '06fd0c9419e813ff5f3fe3a3580a49a810eb8d754fe3ab0944f221e209ad37d3', 'data/raw/background_expansion_20261019/7274d73c8129b4ae466a.source': 'c7828d234a2c068edceab9e1e31d5cbc04caac85a8a4615231ebf453593325ff', 'data/raw/background_expansion_20261019/373d6a0c26e9a313df53.source': 'b8f060b32272e46ffb6c19794fc104ff28851daa434b6095770d0d8efdb97cea', 'data/raw/background_expansion_20261019/b2abbb67b04525dce277.source': '6e3a5553b8052f313941fda8ac7afb6312e58641aee4ed7c8aced5646776898c', 'data/raw/background_expansion_20261019/88b0e9e140cf3429cb97.source': 'fa97a45e5814397060bb19508356cd69aa9b6477ddf840ae0496b1c720c30b84', 'data/raw/background_expansion_20261019/035057e7bafee9c08421.source': '5ef82d89be6445debfde0ed5d5f0070d81f5a25f19a54791fdff57012e0b4bfe', 'data/raw/background_expansion_20261019/3277d033119d1c788bfe.source': '59104fde3579d424d9b629fdeb21e031b9e0c2291c9a4f3266d3822da02e8bbc', 'data/raw/background_expansion_20261019/0c23a5e13cca1b48f154.source': '0ae1a6c3c7700d5ca7cb56eef45bbe43eeb6065496c7c1926bc60c4f8c708ad1', 'data/raw/background_expansion_20261019/31c7ca571e8ec575ea98.source': 'da02c5429bafd39d1859fb3437cd30124ac42980fddead6a162371cb4aebdf85', 'data/raw/background_expansion_20261019/1847f03d1de6f18a683b.source': '89c525dc320add81238c153bb35d8e5acca7bb281c88b7de62c1ccf9b89cd581', 'data/raw/background_expansion_20261019/d7591b3f2a8c8a90e0f0.source': '6a08c278de3247bfa031f76d560f0f9f1ef40c80bb27385bb073d9ba518339c1', 'data/raw/background_expansion_20261019/8af4ff8d7822a5dcb6de.source': '063604e8e313cd940694d70a0744de0d1dbcef261ed3f9bf092efd3d23ea8d46', 'data/raw/background_expansion_20261019/b52012b9e5e7550de2f5.source': '0a02465641001064103cbbb80a45e6132d0991136e8311525f5aa73fe7bda5d5', 'data/raw/background_expansion_20261019/06a729e8f2a6e8dd5b9e.source': '13d98d3897822d0ca5de04681a42337d7d3dc84443f965ae1d8f388f4692ecf0', 'data/raw/background_expansion_20261019/07960149c7b0d5734fb1.source': '3c579ee43ebd63760b446f220734e5e9a6041c7926a09755305e6cc1568a3647', 'data/raw/background_expansion_20261019/ba574160050338785515.source': '89e9bffd737bf1501a0a55d1d77167889fc1ae9021ea1d8bdd97d7ffc3629590', 'data/raw/background_expansion_20261019/bd66678932406109abe6.source': '9427a9b23315effeada0c78fbfc63d55d4823e2d9b462dca0e7e96ccd6daf54d', 'data/raw/background_expansion_20261019/bc3b9dcedd23c668b7e2.source': '4dfa05a7a64c8bb3c328bef0ab0486acaf8778368b0f81319f22e85898369569', 'data/raw/background_expansion_20261019/24b2c6d105ea172f5f0a.source': 'dd16b085769408b7a4af6d170a6aa1709506485453506466dc624546b5c8db64', 'data/raw/background_expansion_20261019/e073b69ca6352e602317.source': '101da34c2d3558ac1e41826bfd479f9e076ad6e805359d3847e2390556e45ffd', 'data/raw/background_expansion_20261019/0108098a8b75f3cadb7a.source': '737d6a547b3012138fccb0e2d1375aea3415d7128d5842ef3f795f28f0bc5755', 'data/raw/council_review_20261019/xu_bulletin_2022.pdf': 'f235d01f3f55f226a269fe4fa002bd2ad068b00d2d34e6c7a97d963e6fa552e5', 'data/raw/background_expansion_20261019/5a9df4305aba6e00ab63.source': '6b80caadfefe6397e93270e076ea99f7d4cbd175a54fc6aecbb726e215c53d9d', 'data/raw/background_expansion_20261019/806bb898ba5b648d29f0.source': 'fc917d512e7aa3c652fb4dae815311f054fc6ea91ac31583d8e72e68f9d616bc', 'data/raw/background_expansion_20261019/8319b9e4fb4b1d382dd2.source': '0371a63d9dce8ac25a791998cf8001020f30d59fe34c28d55210c5485e1a222f', 'data/raw/background_expansion_20261019/07d3192d368be5a7d26c.source': '22ae5353195c3fe1335d3a93ec34a94a52a8ad38b59c7c1ebc6cffc59cce07fe', 'data/raw/background_expansion_20261019/73e37037c5fcfad7bc3b.source': '284eb9f9780e94b50ea685bd704850f1456745762a6c1822b287f778d23d71a7', 'data/raw/background_expansion_20261019/6ea1c714a05afcce6104.source': 'aa34abdc1775871ce150194ba1f62d326ffcab77775afceaa7050758068b3f3a', 'data/raw/background_expansion_20261019/e01e7795c7980972575a.source': '6951956e726dbaa2cd94448ddec8dae42700819d99275942cfc52e0d064bf7ee', 'data/raw/background_expansion_20261019/a836ab3e1f22f8591f85.source': '9c8499711889138b7a5a83daa96c78d98a19a97db16cee8709f9d84dd1b60018', 'data/raw/background_expansion_20261019/91f0fc585b84d4fd9233.source': '4d405c755bf658d2955a136519dfff17eb12537ea4281cc1434926c49c5fbecb', 'data/raw/background_expansion_20261019/b141e9c68065df0bc099.source': 'd6386bdde3161f2247a2e6337aac62b40926e609f7476801b3dbf95dfe3a90b8', 'data/raw/background_expansion_20261019/e8313cf09ed7cf08876a.source': '06d75c6c4d6b94f558cef9f8895c15357b37bd2e67d6b6a24de54f4f4950aad5', 'data/raw/background_expansion_20261019/b69325bd0937649e4ac6.source': '1f118cd0736588230d28e2606ecfb627b487b3026e1e54e5b5e83b55473ce992', 'data/raw/background_expansion_20261019/dfd82c7c6791a2d4a3df.source': '565911eff4d154cf2f8bfab863dec4981d32c62e80b844846a97b0a19805f3d2', 'data/raw/background_expansion_20261019/9fa748f2a94aa8f9a452.source': '315fc8e03910f7a6c3fcde973e2539e06949e55bf6419ecdcfca27781cc091fc', 'data/raw/background_expansion_20261019/02b1d5d956773bc09c19.source': '46e35b34757fdf99bf99b817f7382fbdbd39e17de13653266c5fccec361a26a6', 'data/raw/background_expansion_20261019/b3091e546337a0644adf.source': '47b5da7f8d3c9e83eb94aca79ddd4080c8c30bbde257915bd3f78609bc5580f8', 'data/raw/background_expansion_20261019/e0166f0e2f1ae9722fdd.source': 'eb84987dc2d6789ca34dbac4004ec5408df2993b06625ab5527337bfccb713f4', 'data/raw/background_expansion_20261019/6dcaf031f7d0b9491386.source': '9d68ad2b6b604538af54a523f076fc49b84d29cca276cae3b74ea06be13938a1', 'data/raw/background_expansion_20261019/fb0288d126868c9a4f8c.source': 'e4709a20c605f41475ce07fec0c300ad25da5343891cb8fb8febb5f4a9ca3dcf', 'data/raw/background_expansion_20261019/58abebf1dcb1600612fe.source': '26714a356ea93951f9f2a597dbca78199ec313662e8c7dee8da662764d49d893', 'data/raw/background_expansion_20261019/8e92e6361c8beff974d1.source': '220ad0f9fe34d6eb023ab73c57a2f3ce0b928f74b9e0637fb04eb548ee4da0d1', 'data/raw/background_expansion_20261019/1c13bafc735b3c6e4a67.source': '4fcd2e5c8829df0f1e1ec722702460d2c598392a6e2d3e81882350027f687123', 'data/raw/background_expansion_20261019/c83ee8c4f3a54d8f3fc2.source': 'd52e84805cf631ea2126d5cabaa86755cb6dbec4060837659d26b6c369704b22', 'data/raw/background_expansion_20261019/bec144cc04f51413ece7.source': '85b4f27571c8dfe1f6f731cffcb81345b22e78ae51e5ae318adca4f34e206df3', 'data/raw/background_expansion_20261019/b1f8d7cbadf338e36837.source': '1da68c22a10d4b0e3b70690e21a24fc075d97e4b035313d941f59d92dc8478e3', 'data/raw/background_expansion_20261019/e3ef110f3606c4005dfa.source': 'ea1cfc63d3c173027a8bb2aebb00a3804a5b721140030419646f27712912683b', 'data/raw/background_expansion_20261019/32a12720c01f0c87f272.source': 'cba3b95f5e35ad47d2a360ae526b616fc1b2afd4fb364585031d736abf37052d', 'data/raw/background_expansion_20261019/8f7bfd000b3476d05574.source': '0975db0bd948e9cdad0fff50b7009a896a82276010a29ab14f76c69a734f6bc1', 'data/raw/background_expansion_20261019/d5410342dc5afd961ed9.source': 'd226bbe38206c109c374bf0bdd265bc7ca613266f9e69f6ba27367a00ad3a9e3', 'data/raw/background_expansion_20261019/07ba48c85e8571cc16b8.source': '9459e0a0f93e13ad2cb270b72663bb488848e8d17a3fed8383b10b11978bedd9', 'data/raw/background_expansion_20261019/f39387ca9e3962984830.source': '132b3a73cbb10441e2b62d9316fb1831db862536766e14ea378abb1df4663ad1', 'data/raw/background_expansion_20261019/211d804a52dc5fcc847f.source': '74542ada5d39e030cb34605b0de61bbeaa52cf9a1a907a18e1b94c7d5d636c3e', 'data/raw/background_expansion_20261019/6ca3a982b588f8d9c9e7.source': '254ccc0bee1b274f779e9f60d68a36618777f8b38a14bb8403d12323aff2ffd7', 'data/raw/background_expansion_20261019/267a1681388329f14256.source': 'c4ec4b59e24163526e3504f3d7b798532fafa689f63bcb9ab9e35628d7665151', 'data/raw/background_expansion_20261019/904051aa975f47b74df0.source': 'e7f1785950c4ada99afd9965151616c82261ad09740fab175af37b05cd052684', 'data/raw/background_expansion_20261019/d998f38cf6460c6acee0.source': '79d7c4a07d5817657dfddf890e83885811987ef22d2324be829acb54680ab4a4', 'data/raw/background_expansion_20261019/dacf9b9f67c27011522e.source': '1e4aa0ec32a9e270f21050503637bb5c0e161843b56c1464a6a25e85076da71d', 'data/raw/background_expansion_20261019/4d80937999396e1385c9.source': 'f9177929bf172a53a6f7d38630f1c19c985b304bd020f4eb6e88174b52f1a9db', 'data/raw/background_expansion_20261019/9137491146fa183b3481.source': 'e37a219d08181a96514b1183a7da8b02cd2605617a338a557cff9082f453fb39', 'data/raw/background_expansion_20261019/286f759004ec071a400b.source': 'e8ac54802c8f1532f81908de23da7908dd24f1930aeb98b1b61f8bcfa47d89d4', 'data/raw/background_expansion_20261019/472e0577ac8032157128.source': '32edba7b81b8275dda1e15fb7f48a1771c8ae1bb8802e2bb6316cfa24cb26462', 'data/raw/background_expansion_20261019/808aef947463a8d05682.source': 'f1eaf755297769c3be8056c326cad006c1ee134c7dced0ba6a92e75bc35bac8c', 'data/raw/background_expansion_20261019/2bc6b319e633387c421b.source': '8a3a2ae3e6578387f4f769b13cd051428d4253d33509dc588df4cd0457d2bc34', 'data/raw/background_expansion_20261019/2336440b33ef9bf8e0dc.source': 'a1dd34c42d0ac3e3a10514643b58b4f1d7fa9d98185239f1ddb20dc2ff77017a', 'data/raw/background_expansion_20261019/d737c42abcdfc7800c0c.source': '1d4b30b795148e99639d2b3015d22e5efc92a5fc151ef97aee2e10fc60fbcb28', 'data/raw/background_expansion_20261019/b4710d536a4f3006a8e8.source': '1599c5350e341fbaeaa62fe523b84561d23814925ecee26b10e9587c778a1ff1', 'data/raw/background_expansion_20261019/5fe08ff2ea97f2adebc2.source': '458b1818712866dfa98907c935bdb0cdefb870c6c1798e9be062e0d13df0322b', 'data/raw/background_expansion_20261019/476f80b52eabb5030cf1.source': 'd363c4535389df0cf340090c6a4480de09f58ec4ca3d9555cbdc1f90abad7092', 'data/raw/background_expansion_20261019/dc5bbe3a0bcf1f690b18.source': '572b28cbd432dc885b65c4ae7795fd31f0ab8e94c5a56742cc0429801333232d', 'data/raw/background_expansion_20261019/a57b3dfaabca28d5397c.source': '4ea7c6b7d5b6891939b5a9942eff01314e1d20c66eb07c5eb33e08d9029b7bb7', 'data/raw/background_expansion_20261019/77a7e4e8f4d8ec16264b.source': '5f696537f91e048b665fb13ed86c274cea99474d5e162be0a9b734ea0fd69156', 'data/raw/background_expansion_20261019/cbda509cb932ff0f177e.source': 'c33d0b851ae1bededd3e090ff90ddf9ba5d72889574a341feedffc7daa5d309b', 'data/raw/background_expansion_20261019/ae578ccaadcf905067d6.source': '48fdb00b1c1069786cfcf167810b71c53ae62d613b1405bfb2074287d9d16940', 'data/raw/background_expansion_20261019/c58078f0925ce33a3646.source': '9ea9291c96e3b795291e05b8a8fd2d506d400087c86d6e5c119d3c75fcb103f9', 'data/raw/background_expansion_20261019/960cd0171a4ec4071ee1.source': '3350987e11f57af7697acb4be9fb6b77a992189b6590a381433b725cc11be2d3', 'data/raw/background_expansion_20261019/1fb255bf0b532d181954.source': 'e0e9bd5797dd8250ed88f809df97f6d91e22f0386ad52c963e6cdeaa39283665', 'data/raw/background_expansion_20261019/37db3e3d881c5aa7ebd6.source': '6067e2c863077c2df6ee21751cd8f499154eefaf10020926d8233f1da157a9e4', 'data/raw/background_expansion_20261019/b468c487d8942ab8165a.source': '1593d436792775270505151a6fd3dc8db4846f62936657b76a0f79ff4460c647', 'data/raw/background_expansion_20261019/2427170889bf4cd0c185.source': 'ff0dd6dbe72a6322b6a2b0d543d3e53f7a2a48098f0a658d863f468be7939219', 'data/raw/background_expansion_20261019/909956268dd7756c7d9f.source': 'f688e42b05d123f76107462c5276a477dcebea136cffe2c8710bfe678be75474', 'data/raw/background_expansion_20261019/191cdf84ae7775682dea.source': 'a42bbc93c1a4c0bd3ebeddbe82e3ab54f64dca30648eeb937e496b0a3c3510d7', 'data/raw/background_expansion_20261019/3f9e9546e799451ea126.source': '012ed395f817db6ea04b4d77da3e25c83dba356df1429d53e2917926b646379f', 'data/raw/background_expansion_20261019/0eaed98513b0dd4f1ae6.source': '4ba79df71ac92362c5956cbe516e5079426d6c390c0e1b213bec6c410a876ad2', 'data/raw/background_expansion_20261019/8bd1282f575cbf4969f0.source': 'eeca7c5bc9f1cb199113d9d1c60dbc89d2ebc177a66666d5d5de5d22f5a9f149', 'data/raw/background_expansion_20261019/05df9c9e6afda280835b.source': 'a89f54ba23e16d3025ec81a6fad4e11ad779037ed4a177d1dbcff87767b26047', 'data/raw/background_expansion_20261019/5bb27d08fa3fde9c20a6.source': 'bae1060a8e85d84ab1c489cbe4f7a129e06ae0dd5b03bd97d143a474c717a5f1', 'data/raw/background_expansion_20261019/3da3f56863d3827c17dc.source': '003de984b365357e9ff55ff9bf0987ee091a3502ae7249807380522ab9df2d0e', 'data/raw/background_expansion_20261019/de42f489a313a6dfe21d.source': 'ed69b67e4e9a2b4a763f5f5c2f7f1c8a2466edfd31e6e89f32ec96368beb4e21', 'data/raw/cec_all_offices_manifest.json': '9cc85f7f913f5de717ba92ec7698c73cedb29df506fba59a69933d1ac3517231', 'data/raw/cec_all_offices/068_elcand.csv': '47ee3b569c7bd6d2d0e989d087457a10dc4a7c2657df51c9c1c97a6436f96720', 'data/raw/cec_all_offices/080_elcand.csv': '19cfbfb2e639ff2e5310856ec64e7933b6d4ff483dc09bd73092075ba12db6e7', 'data/raw/cec_all_offices/095_elcand.csv': '2106bc9400f7d64784595799d2404496186cffaa56d82b14554d172d781f1b60', 'data/raw/cec_all_offices/116_elcand.csv': 'cf9a675fd5d26c6a6f2961f3c0654699c0fe7905c81a2e68d26d9f3cd852ee3d', 'data/raw/sex_verification/chen_zhongjia_retry.html': 'c10bbe149bb92452cb1793ef3fe19d3483cb14254765768fbb1fee314c732c37', 'data/raw/sex_verification/capture_manifest_2.json': 'f0149f02a44191d0c2d0b4540ea2501a9d21070b8bd40c39c46d7fe7bfd9cf55', 'data/raw/sex_verification/wen_shizheng.pdf': '18316813b98dcf6a59c1afe8f194ad9377515e94ce5d54f26d9a3eae2cf1025c', 'data/raw/mayor_birth_followup_20261019/manifest.json': 'c6bb81a76aa0641236ab21dd10f6085731756a1d75dc1808a7c61c4ab543634c', 'data/raw/mayor_birth_followup_20261019/wen_cna.html': 'c996c0469ef05ff6fdd1f7cefbc3f7e109bb21969de42fcf591498a828a7ad5f', 'data/raw/mayor_birth_followup_20261019/wen_nownews.html': 'e25c358cf17143a8055b5acf2cf676669b4aac553d128df421d502f01440d8a4', 'data/raw/mayor_birth_followup_20261019/web_tool_excerpts.json': '82a726fcc4a7ffe05c812300cb87d488b7683bfc412188313dcde404440dedf2'}
for p,h in PINS.items():
 q=(ROOT/p).resolve();assert q.is_relative_to(ROOT.resolve()) and q.is_file() and hashlib.sha256(q.read_bytes()).hexdigest()==h,p
print("Verified exact source / metadata files:",len(PINS))

Verified exact source / metadata files: 504


## Change for Your Own Data

本文件只讀取上述固定來源，不重新下載、不整理背景欄位、不覆寫原檔。

In [2]:
MANIFESTS=['data/raw/social_followup_round6_20261019/manifest.json', 'data/raw/career_expansion_20261019/manifest.json', 'data/raw/mayor_incumbency_20261019/manifest.json', 'data/raw/mayor_birth_followup_20261019/manifest.json']
records=[]
for p in MANIFESTS:records+=json.loads((ROOT/p).read_text())
df=pd.DataFrame(records).fillna("")
assert len(df)==262
print("Capture attempts",len(df),"distinct source URLs",df.url.nunique())

Capture attempts 262 distinct source URLs 261


## Summary

### Sample

In [3]:
display(df[["url","status","path"]].head(10))

,url,status,path
0,https://youth.pfpnext.com/2026/candidate-detai...,200,data/raw/social_followup_round6_20261019/c9fb4...
1,https://youth.pfpnext.com/2026/candidate-detai...,200,data/raw/social_followup_round6_20261019/fff69...
2,https://youth.pfpnext.com/2026/candidate-detai...,200,data/raw/social_followup_round6_20261019/b39ba...
3,https://youth.pfpnext.com/2026/candidate-detai...,200,data/raw/social_followup_round6_20261019/ce69d...
4,https://youth.pfpnext.com/2026/candidate-detai...,200,data/raw/social_followup_round6_20261019/02779...
5,https://youth.pfpnext.com/2026/candidate-detai...,200,data/raw/social_followup_round6_20261019/048db...
6,https://youth.pfpnext.com/2026/candidate-detai...,200,data/raw/social_followup_round6_20261019/a201e...
7,https://youth.pfpnext.com/2026/candidate-detai...,200,data/raw/social_followup_round6_20261019/f6a94...
8,https://youth.pfpnext.com/2026/candidate-detai...,200,data/raw/social_followup_round6_20261019/f5032...
9,https://youth.pfpnext.com/2026/candidate-detai...,200,data/raw/social_followup_round6_20261019/dd2a1...


### Summary Stats

In [4]:
display(df.status.value_counts(dropna=False).rename_axis("HTTP status").to_frame("attempts"));print("Saved body files",sum(bool(a.get("path")) for a in records));print("Raw bytes checked; no claim of complete candidate fields.")

,attempts
HTTP status,
200,227
000,16
403,15
406,3
404,1


Saved body files 246
Raw bytes checked; no claim of complete candidate fields.


2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。